# Exploración: SUNARP — Servicio de Publicidad Registral en Línea (SPRL)

- **Entidad oficial:** Superintendencia Nacional de los Registros Públicos (SUNARP).
- **Descripción y propósito:** Plataforma transaccional autenticada del Estado peruano que permite acceder a los servicios de publicidad registral, búsqueda de índices y visualización de partidas registrales (Personas Jurídicas, Propiedad Inmueble, Mandatos y Poderes, etc.).
- **Enlaces del flujo SSO:**
  - Portada de ingreso: `https://sprl.sunarp.gob.pe/sprl/ingreso`
  - Pasarela de autorización (SSO): `https://im01-autorizacion-sprl-production.apps.paas.sunarp.gob.pe/v1/sunarp-services/im/autorizacion/login`
  - Módulo objetivo (Visualización de partidas): `https://sprl.sunarp.gob.pe/sprl/main/partidas-base-grafica-registral`
- **Mecanismo de acceso:** Autenticación mediante cuenta de usuario registrada (`SUNARP_USER` y `SUNARP_PASS`) protegida con desafío interactivo de **Cloudflare Turnstile**.
- **Resultado del módulo de autenticación:** Obtención y persistencia en disco (`sunarp_sprl_auth.json` + perfil nativo `./chrome_sprl_profile`) de las cookies de sesión (`cf_clearance`, cookies de aplicación) y del estado de `localStorage` (tokens JWT de los orígenes de SUNARP) para reutilizarlos en consultas posteriores sin repetir el login.

## Estrategia Técnica de Autenticación y Evasión Anti-Bot

- **Navegador con contexto persistente (`launch_persistent_context`):**
  1. Se levanta Google Chrome nativo (`channel="chrome"`) sobre un directorio de perfil dedicado (`./chrome_sprl_profile`), sin sobrescribir el `User-Agent` real del binario para mantener consistencia total con los *Client Hints* (`sec-ch-ua`) evaluados por Cloudflare.
  2. Se desactiva la bandera `--no-sandbox` por defecto de Playwright (`ignore_default_args=["--no-sandbox"]`) y se oculta `navigator.webdriver`.
- **Orquestación del flujo Single Sign-On (SSO):**
  1. Se navega inicialmente a la ruta privada objetivo (`/sprl/main/partidas-base-grafica-registral`). Si no existe sesión activa, el portal redirige a `/sprl/ingreso`.
  2. Se cierra automáticamente el modal publicitario emergente (*"Sunarp Simplifica"*) mediante evento `Escape`, clic en el botón de cierre o limpieza del contenedor overlay en el DOM, y se acciona el botón `INGRESAR` de la portada para generar el contexto de redirección SSO válido.
- **Resolución automatizada de Cloudflare Turnstile:**
  1. Se completan las credenciales con tipeo secuencial (`press_sequentially`) simulando cadencia humana.
  2. Se monitorea el `input[name="cf-turnstile-response"]` y, en caso de requerir interacción sobre la casilla *"Verifique que es un ser humano"*, se calcula dinámicamente la caja delimitadora (`bounding_box`) del widget para ejecutar un clic físico con trayectoria de ratón (`page.mouse.move` + `page.mouse.click`).
- **Persistencia de estado (`storage_state`):**
  - Una vez completada la redirección automática hacia `/sprl/main/partidas-base-grafica-registral`, se exportan las cookies y los orígenes (`localStorage`) a `sunarp_sprl_auth.json`.

In [ ]:
import asyncio
import json
import os
import re
from pathlib import Path
from io import BytesIO
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
import pytesseract
from playwright.async_api import async_playwright
import base64

# Detección automática de entorno (Windows vs Docker/Linux)
if os.name == 'nt':
    pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# 1. Configuración General
URL_OBJETIVO_SPRL = "https://sprl.sunarp.gob.pe/sprl/main/partidas-base-grafica-registral"
URL_LOGIN_DIRECTO = "https://im01-autorizacion-sprl-production.apps.paas.sunarp.gob.pe/v1/sunarp-services/im/autorizacion/login"

SUNARP_USER = os.getenv("SUNARP_USER", "MYUSER123")
SUNARP_PASS = os.getenv("SUNARP_PASS", "BU&VQESCG9")

PARAM_OFICINA = "CUSCO"
PARAM_AREA = "Personas Jurídicas"
PARAM_PARTIDA = "11156404"

ARCHIVO_SESION = Path("sunarp_sprl_auth.json")
PERFIL_DIR = Path("./chrome_sprl_profile")

# 2. Funciones Auxiliares
def resolver_captcha_imagen(imagen_bytes: bytes) -> str:
    """Preprocesamiento para Tesseract sin binarización destructiva."""
    img = Image.open(BytesIO(imagen_bytes)).convert("L")
    
    # 1. Invertimos para tener letras oscuras sobre fondo claro
    img = ImageOps.invert(img)
    
    # 2. Autocontraste suave (NO usamos img.point porque une la línea con la letra)
    img = ImageOps.autocontrast(img, cutoff=2)
    
    # 3. CRÍTICO PARA TESSERACT: Agregar un borde blanco de 20px alrededor
    img = ImageOps.expand(img, border=20, fill=255)
    
    # 4. Escalar x2 con filtro bicúbico
    w, h = img.size
    img = img.resize((w * 2, h * 2), Image.Resampling.BICUBIC)
    
    # PSM 8 trata la imagen como una sola palabra continua
    custom_config = r'--oem 3 --psm 8 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
    texto = pytesseract.image_to_string(img, config=custom_config)
    return re.sub(r'[^A-Z0-9]', '', texto.upper())

async def cerrar_modal_publicitario(page):
    await page.wait_for_timeout(1000)
    await page.keyboard.press("Escape")
    await page.wait_for_timeout(400)
    selectores_x = ["nz-modal-close", "button[aria-label*='Close' i]", "span:has-text('×')"]
    for sel in selectores_x:
        loc = page.locator(sel).first
        if await loc.count() > 0 and await loc.is_visible():
            try:
                await loc.click(timeout=1000)
                await page.wait_for_timeout(500)
                return
            except Exception:
                pass
    await page.mouse.click(15, 15)

async def clicar_checkbox_turnstile(page) -> bool:
    selectores = ["iframe[src*='cloudflare']", "iframe[src*='turnstile']", "div.cf-turnstile"]
    for sel in selectores:
        loc = page.locator(sel).first
        if await loc.count() > 0 and await loc.is_visible():
            box = await loc.bounding_box()
            if box:
                x = box["x"] + 32
                y = box["y"] + (box["height"] / 2)
                await page.mouse.move(x, y, steps=10)
                await page.wait_for_timeout(150)
                await page.mouse.click(x, y)
                return True
    return False

# 3. Flujo Principal Unificado
async def consulta_sunarp_unificada() -> dict:
    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            user_data_dir=str(PERFIL_DIR),
            headless=False,
            channel="chrome",
            viewport={"width": 1366, "height": 768},
            locale="es-PE",
            ignore_default_args=["--no-sandbox"],
            args=["--disable-blink-features=AutomationControlled"]
        )
        await context.add_init_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        page = context.pages[0] if context.pages else await context.new_page()

        try:
            # === FASE A: LOGIN ===
            print(f"[1/6] Evaluando estado de sesión en: {URL_OBJETIVO_SPRL}...")
            await page.goto(URL_OBJETIVO_SPRL, wait_until="domcontentloaded", timeout=30000)
            
            for _ in range(20):
                texto_body = await page.inner_text("body")
                if "sprl/ingreso" in page.url or "autorizacion" in page.url or "Hola," in texto_body or await page.locator("input[type='password']").count() > 0:
                    break
                await page.wait_for_timeout(500)

            if "Hola," not in await page.inner_text("body") and "Búsqueda Directa" not in await page.inner_text("body"):
                print("[2/6] Sesión no detectada. Iniciando flujo de login SSO...")
                if "sprl/ingreso" in page.url:
                    await cerrar_modal_publicitario(page)
                    btn_portada = page.locator("button:has-text('INGRESAR'), a:has-text('INGRESAR')").first
                    if await btn_portada.count() > 0:
                        await btn_portada.click()
                        await page.wait_for_load_state("domcontentloaded")
                        await page.wait_for_timeout(1500)

                input_user = page.locator("input[placeholder*='usuario' i], input[type='text']").first
                input_pass = page.locator("input[placeholder*='contraseña' i], input[type='password']").first
                await input_user.wait_for(state="visible", timeout=15000)
                await input_user.fill(SUNARP_USER)
                await input_pass.fill(SUNARP_PASS)

                print("   -> Resolviendo Turnstile de Login...")
                await page.wait_for_timeout(1000)
                for intento in range(15):
                    if await page.evaluate("() => { const el = document.querySelector('input[name=\"cf-turnstile-response\"]'); return el ? el.value : ''; }"):
                        break
                    if intento in (2, 6, 10):
                        await clicar_checkbox_turnstile(page)
                    await page.wait_for_timeout(1000)

                await page.locator("button:has-text('INGRESAR')").first.click()
                await page.locator("text=/Hola,/i").first.wait_for(state="visible", timeout=25000)

            if "partidas-base-grafica-registral" not in page.url:
                await page.goto(URL_OBJETIVO_SPRL, wait_until="domcontentloaded", timeout=30000)
            
            await page.locator("text=Búsqueda Directa por Número de Partida").first.wait_for(state="visible", timeout=20000)
            await page.wait_for_timeout(1000)
            await context.storage_state(path=str(ARCHIVO_SESION))

            # === FASE B: EXTRACCIÓN Y BÚSQUEDA ===
            print("[3/6] Resolviendo Cloudflare Turnstile interno...")
            for intento in range(15):
                if await page.evaluate("() => { const el = document.querySelector('input[name=\"cf-turnstile-response\"]'); return el ? el.value : ''; }"):
                    break
                if intento in (2, 5, 8):
                    await clicar_checkbox_turnstile(page)
                await page.wait_for_timeout(1000)

            print(f"[4/6] Seleccionando Oficina: {PARAM_OFICINA} y Área: {PARAM_AREA}...")
            cajas_select = page.locator("nz-select")
            await cajas_select.first.wait_for(state="visible", timeout=15000)

            # 1. Oficina
            print("   -> Desplegando Oficina Registral...")
            await cajas_select.nth(0).click()
            await page.wait_for_timeout(1000)
            
            await page.keyboard.type(PARAM_OFICINA, delay=50)
            await page.wait_for_timeout(800)
            
            opc_oficina = page.locator(f"nz-option-item[title*='{PARAM_OFICINA}' i]").first
            if await opc_oficina.count() > 0:
                await opc_oficina.evaluate("el => el.click()")
            else:
                await page.keyboard.press("Enter")
            await page.wait_for_timeout(1000)

            # 2. Área
            print("   -> Desplegando Área Registral...")
            await cajas_select.nth(1).click()
            await page.wait_for_timeout(1000)
            
            param_area_safe = "Personas Jur" 
            await page.keyboard.type(param_area_safe, delay=50)
            await page.wait_for_timeout(800)
            
            opc_area = page.locator(f"nz-option-item[title*='{param_area_safe}' i]").first
            if await opc_area.count() > 0:
                await opc_area.evaluate("el => el.click()")
            else:
                await page.keyboard.press("Enter")
                
            await page.wait_for_timeout(2500) 

            # 3. Radio Partida (Clic físico por coordenadas)
            print("   -> Seleccionando opción 'Partida'...")
            radio_partida = page.locator("label:has-text('Partida')").first
            await radio_partida.wait_for(state="visible", timeout=5000)
            box = await radio_partida.bounding_box()
            if box:
                # Simulamos un clic real del mouse en el centro de la palabra "Partida"
                await page.mouse.click(box["x"] + (box["width"] / 2), box["y"] + (box["height"] / 2))
            else:
                await radio_partida.click(force=True)
                
            await page.wait_for_timeout(1500)

            # 4. Número
            print(f"   -> Ingresando número de partida: {PARAM_PARTIDA}...")
            input_numero = page.locator("#numero").first
            await input_numero.wait_for(state="visible", timeout=5000)
            await input_numero.click()
            await page.keyboard.press("Control+A") 
            await page.keyboard.press("Backspace")
            await input_numero.fill("")
            await input_numero.press_sequentially(PARAM_PARTIDA, delay=50)
            await input_numero.press("Tab")  
            await page.wait_for_timeout(500)

            print("[5/6] Resolviendo CAPTCHA OCR (Modo 6-Dígitos)...")
            contenedor_captcha = page.locator("app-captcha-sunarp").first
            img_captcha_locator = contenedor_captcha.locator("img.img-captcha").first
            input_captcha = contenedor_captcha.locator("input").first
            btn_recargar = contenedor_captcha.locator("a[color='warn']").first
            btn_buscar = page.locator("button:has-text('Buscar')").first

            max_reintentos = 15
            tabla_encontrada = False

            for intento in range(1, max_reintentos + 1):
                await page.wait_for_timeout(1500) # Esperar a que renderice la imagen
                await img_captcha_locator.wait_for(state="visible", timeout=10000)
                
                img_bytes = await img_captcha_locator.screenshot()
                texto_captcha = resolver_captcha_imagen(img_bytes)
                print(f"   -> [Intento {intento}/{max_reintentos}] OCR detectó: '{texto_captcha}'")

                # REGLA ESTRICTA: El captcha SIEMPRE tiene 6 caracteres
                if len(texto_captcha) != 6:
                    print(f"   -> Longitud incorrecta ({len(texto_captcha)}/6). Recargando...")
                    if await btn_recargar.is_visible(): 
                        await btn_recargar.click(force=True)
                    continue

                # Escribir el valor
                await input_captcha.click()
                await page.keyboard.press("Control+A") 
                await page.keyboard.press("Backspace")
                await input_captcha.fill("")
                
                await input_captcha.press_sequentially(texto_captcha, delay=50)
                await input_captcha.press("Enter")
                await page.wait_for_timeout(500)
                
                await btn_buscar.click(force=True)

                try:
                    respuesta = await page.wait_for_selector(".ant-table, table, nz-modal-confirm-container, .ant-modal-wrap, .ant-message", timeout=6000)
                    clase_res = await respuesta.get_attribute("class") or ""
                    tag_name = await respuesta.evaluate("el => el.tagName.toLowerCase()")
                    
                    if "modal" in clase_res or "modal" in tag_name:
                        print("   -> Error SUNARP (Modal Detectado). Cerrando modal...")
                        btn_aceptar = page.locator("button:has-text('Aceptar'), button:has-text('OK')").first
                        if await btn_aceptar.is_visible():
                            await btn_aceptar.click()
                        # IMPORTANTE: No damos clic a recargar, esperamos que el sistema cambie la imagen por sí solo
                        await page.wait_for_timeout(2000)
                            
                    elif "message" in clase_res or "notification" in clase_res:
                        print("   -> Error SUNARP (Alerta pequeña). Recargando manualmente...")
                        if await btn_recargar.is_visible(): 
                            await btn_recargar.click(force=True)
                        await page.wait_for_timeout(1000)
                            
                    else:
                        print("   -> ¡Captcha correcto! Tabla cargada exitosamente.")
                        tabla_encontrada = True
                        break
                        
                except Exception:
                    # En caso de timeout, forzamos recarga
                    if await btn_recargar.is_visible(): 
                        await btn_recargar.click(force=True)

            if not tabla_encontrada:
                return {"error": "Se superó el límite de reintentos del CAPTCHA OCR sin éxito."}

            print("[6/6] Extrayendo datos de la tabla...")
            await page.wait_for_timeout(1000)
            filas_locator = page.locator("table tbody tr")
            resultados = []

            for i in range(await filas_locator.count()):
                celdas = filas_locator.nth(i).locator("td")
                if await celdas.count() >= 8:
                    resultados.append({
                        "zona_registral": await celdas.nth(0).inner_text(),
                        "oficina_registral": await celdas.nth(1).inner_text(),
                        "partida": await celdas.nth(2).inner_text(),
                        "numero_ficha": await celdas.nth(3).inner_text(),
                        "tomo": await celdas.nth(4).inner_text(),
                        "folio": await celdas.nth(5).inner_text(),
                        "area_registral": await celdas.nth(6).inner_text(),
                        "registro_de": await celdas.nth(7).inner_text(),
                    })

            return {
                "audit": {
                    "source": "SUNARP_SPRL",
                    "oficina": PARAM_OFICINA,
                    "partida": PARAM_PARTIDA,
                    "status": "SUCCESS" if resultados else "NO_DATA"
                },
                "data": resultados
            }

        except Exception as exc:
            return {"error": str(exc)}
        finally:
            await context.close()

# Ejecución de la celda única
resultado_extraccion = await consulta_sunarp_unificada()
print("\n================== RESULTADO FINAL SUNARP ==================")
print(json.dumps(resultado_extraccion, indent=2, ensure_ascii=False))

[1/6] Evaluando estado de sesión en: https://sprl.sunarp.gob.pe/sprl/main/partidas-base-grafica-registral...
[3/6] Resolviendo Cloudflare Turnstile interno...
[4/6] Seleccionando Oficina: CUSCO y Área: Personas Jurídicas...
   -> Desplegando Oficina Registral...
   -> Desplegando Área Registral...
   -> Seleccionando opción 'Partida'...
   -> Ingresando número de partida: 11156404...
[5/6] Resolviendo CAPTCHA OCR (Modo 6-Dígitos)...
   -> [Intento 1/15] OCR detectó: 'ZW'
   -> Longitud incorrecta (2/6). Recargando...
   -> [Intento 2/15] OCR detectó: 'SGZ7B'
   -> Longitud incorrecta (5/6). Recargando...
   -> [Intento 3/15] OCR detectó: 'JGEZ'
   -> Longitud incorrecta (4/6). Recargando...
   -> [Intento 4/15] OCR detectó: '3GNG42'
   -> Error SUNARP (Modal Detectado). Cerrando modal...
   -> [Intento 5/15] OCR detectó: 'SZVBPH'
   -> Error SUNARP (Modal Detectado). Cerrando modal...
   -> [Intento 6/15] OCR detectó: 'H64F'
   -> Longitud incorrecta (4/6). Recargando...
   -> [Intento